# Structured Output
We can ask the model to give output in a specified schema.The structure can be used for processing in later stages


In [21]:
import os
from langchain.chat_models import init_chat_model
from dotenv  import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7ff14d5b1a20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7ff14ec28be0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [22]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="The year movie got released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")



In [23]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7ff14d5b1a20>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7ff14ec28be0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year movie got released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movies rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'r

In [24]:
## lets compare the output
response_unstruct=model.invoke("Provide me details about inception")
print(response_unstruct.content)
response_struct=model_with_structure.invoke("Provide me details about movie Inception")
print("Structured output----")
print(response_struct)

<think>
Okay, the user wants details about "Inception." First, I should confirm if they're referring to the movie or the concept. Since "Inception" is a well-known movie, I'll start with that. I need to cover the basics: director, cast, release year, genre. Then, the plot summary is essential. Maybe break it down into the premise and key elements like the dream-sharing tech. I should mention the main characters and their roles. Christopher Nolan wrote and directed it, so that's important. The cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, etc. The themes are significant too—reality, dreams, time, identity. The user might also be interested in critical reception, awards, and maybe some behind-the-scenes info. Oh, and the ending is famous for ambiguity, so explaining that could be helpful. I should also check if there's a connection to the concept of inception in general, like the idea of planting an idea, but the user might just want the movie details. Let me structu

In [25]:
#Message output with parsed structure
class Movie(BaseModel):
    # ... ->tells that it is a required field
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year movie got released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The movies rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)

response=model_with_structure.invoke("Provide me details about the Movie Robo")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Robo." Let me see what I need to do here. The available tool is the Movie function, which requires title, year, director, and rating. The user didn\'t specify the year, director, or rating, so I don\'t have that information. Wait, but the function requires all four parameters. Hmm, how can I get that data? Maybe I should check if there\'s a movie titled "Robo" in my knowledge base.\n\nI recall there\'s a Tamil movie called "Robo" released in 2017, directed by Raja Krishna Meyer. The lead actor is Rajinikanth. The rating might be around 7.5 or 8 on IMDb. Let me confirm the details. Title: Robo, Year: 2017, Director: Raja Krishna Meyer, Rating: 7.5/10. That seems right. I should use these details to construct the function call with the provided parameters. Make sure all required fields are included. Yeah, that\'s covered. Let me format the JSON correctly.\n', 'tool

In [26]:
## Nested structre

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float| None=Field(None,description="Budget in million dollars")


model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide me details about the Movie Inception")
response



MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160.0)

In [27]:
#TypedDict -- it dont do any rutime validation

from typing_extensions import TypedDict,Annotated
# we will get a plain dictionary as output structure
class MovieDict(TypedDict):
    """ A movie with details"""
    title: Annotated[str,...,"The title of the movie"]
    year: Annotated[int,...,"the year movie was released"]
    director:Annotated[str,...,"the director of the movie"]
    rating:Annotated[float,...,"The movies rating out of 10"]

model_withtypeddict=model.with_structured_output(MovieDict)
response=model_withtypeddict.invoke("Please provide me details about Avatar")
response

{'director': 'James Cameron', 'rating': 7.8, 'title': 'Avatar', 'year': 2009}

In [28]:
## Nested structre

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float| None=Field(None,description="Budget in million dollars")


model_with_structure=model.with_structured_output(MovieDetails)

response=model_with_structure.invoke("Provide me details about the Movie Avatar")
response

{'budget': 237000000,
 'cast': [{'name': 'Sam Worthington', 'role': 'Jake Sully'},
  {'name': 'Zoe Saldaña', 'role': 'Neytiri'},
  {'name': 'Sigourney Weaver', 'role': 'Dr. Grace Augustine'},
  {'name': 'Stephen Lang', 'role': 'Colonel Miles Quaritch'},
  {'name': 'Michelle Rodriguez', 'role': 'Trudy Chacon'}],
 'genres': ['Science Fiction', 'Action', 'Adventure'],
 'title': 'Avatar',
 'year': '2009'}

In [29]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

In [34]:
# DataClass is used just to hold the state (data),no complex logics

from langchain.agents import create_agent
class ContactInfo(BaseModel):
    name:str=Field(description="The name of the person")
    email:str=Field(description="Email of the person")
    phone:str=Field(description="The phone number of the person")

model=init_chat_model("groq:qwen/qwen3-32b")
agent=create_agent(
    model=model,
    response_format=ContactInfo 
)

result_basemodel=agent.invoke(
    {"messages":[
        {"role":"user",
         "content":"Extract the contact info from below:Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654"}

        ]}
)

result_basemodel


{'messages': [HumanMessage(content='Extract the contact info from below:Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654', additional_kwargs={}, response_metadata={}, id='2229c3af-3519-415d-b99a-98ed853c11b1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The input is "Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654". \n\nFirst, I need to identify the different parts here. The name is probably "Nithya Sai" since that\'s at the beginning and looks like a full name. Then there\'s an email address: "nithyasaitejareddy99@gmail.com". The last part is "Contact Info 999987654" – the phrase "Contact Info" might be a label for the phone number, so the phone number here would be 999987654.\n\nLooking at the tool provided, the ContactInfo function requires name, email, and phone. All three are present here. The phone number is a string of digits, which is 

In [35]:
## TypedDict output
from langchain.agents import create_agent
class ContactInfo(TypedDict):
    name:str=Field(description="The name of the person")
    email:str=Field(description="Email of the person")
    phone:str=Field(description="The phone number of the person")

model=init_chat_model("groq:qwen/qwen3-32b")
agent=create_agent(
    model=model,
    response_format=ContactInfo 
)

result_typeddict=agent.invoke(
    {"messages":[
        {"role":"user",
         "content":"Extract the contact info from below:Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654"}

        ]}
)

result_typeddict


{'messages': [HumanMessage(content='Extract the contact info from below:Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654', additional_kwargs={}, response_metadata={}, id='392f08ed-3c20-4cf9-8c48-f867e9fc7ef8'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let\'s see. The user wants me to extract contact information from the given text. The text provided is "Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654". \n\nFirst, I need to identify the different parts of the contact info. The name is probably "Nithya Sai" since that\'s at the beginning and looks like a person\'s name. Next, there\'s an email address: "nithyasaitejareddy99@gmail.com". That\'s straightforward because it has the typical structure of an email with a domain.\n\nThen there\'s the phone number part. The text says "Contact Info 999987654". The phrase "Contact Info" might be a label, so the number following it, 999987654, is likely the phone number. However, phone

In [38]:
# we will use the Dataclass - no __init__ fun required
from langchain.agents import create_agent
from dataclasses import dataclass

@dataclass
class ContactInfo():
    name:str=Field(description="The name of the person")
    email:str=Field(description="Email of the person")
    phone:str=Field(description="The phone number of the person")

model=init_chat_model("groq:qwen/qwen3-32b")
agent=create_agent(
    model=model,
    response_format=ContactInfo 
)

result_dataclass=agent.invoke(
    {"messages":[
        {"role":"user",
         "content":"Extract the contact info from below:Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654"}

        ]}
)

result_dataclass


{'messages': [HumanMessage(content='Extract the contact info from below:Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654', additional_kwargs={}, response_metadata={}, id='f35c7432-e0bc-449c-a7f9-3b1c1d2418be'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, let me see. The user wants me to extract the contact information from the given text. The text provided is "Nithya Sai, nithyasaitejareddy99@gmail.com,Contact Info 999987654". \n\nFirst, I need to identify the different parts of the contact info. The name is probably "Nithya Sai" since that\'s at the beginning and looks like a full name. Next, the email address is clearly "nithyasaitejareddy99@gmail.com" as it follows the standard email format. \n\nThen there\'s the phone number part. The text says "Contact Info 999987654". The number here is 999987654. But wait, phone numbers usually have more digits. Let me check if there\'s a typo or if maybe the number is written without proper formatting